In [27]:
from probabilistic_word_embeddings.estimation import map_estimate 
from probabilistic_word_embeddings.embeddings import Embedding
from simulate import  load_simulated_data_generator, load

import numpy as np
def sufficient_statistics(data, vocabulary):
    V = len(vocabulary)
    count_matrix_x1 = np.zeros((V, V), dtype=int)
    count_matrix_total = np.zeros((V, V), dtype=int)

    for item in data:
        v_index = vocabulary[item['v']]
        w_index = vocabulary[item['w']]
        if item['x'] == 1:
            count_matrix_x1[v_index, w_index] += 1
        #elif item['x'] == 0:
        #    count_matrix_x0[v_index, w_index] += 1
        count_matrix_total[v_index, w_index] += 1

    return count_matrix_x1, count_matrix_total


def sigmoid(x:float): # might be a good idea to have this in a common library.
  return 1 / (1 + np.exp(-x))
def compare_statistics(v, w, theta, counts_1, counts_total, vocabulary):

    V=  len(vocabulary)

    if v == w:
        return
    vi, wi = vocabulary[v], vocabulary[w]
    
    rho_v = theta[vi, :]          # $\rho_v = \theta_v$
    alpha_w = theta[wi + V, :]    # $\alpha_w = \theta_{V + w}$
    eta = rho_v.dot(alpha_w)      # $\eta = \rho_v^T \alpha_w$
    p = sigmoid(eta)              # Run it through the link function $\sigma; p = \sigma(\eta)$

    observed_ratio = counts_1[vi, wi] / counts_total[vi, wi] if counts_total[vi, wi] > 0 else 0.0

    print(f'{v}, {w}')
    print('from theta :', p)
    print('observed   :', observed_ratio)
    #return p, observed_ratio

In [2]:


file_path = 'test100k.json'
for item in load_simulated_data_generator(file_path, batch_size=5):
    print(item)  
    break

(<tf.Tensor: shape=(5,), dtype=string, numpy=
array([b'word_4', b'word_8', b'word_4', b'word_7', b'word_7'],
      dtype=object)>, <tf.Tensor: shape=(5,), dtype=string, numpy=
array([b'word_5', b'word_9', b'word_7', b'word_6', b'word_9'],
      dtype=object)>, <tf.Tensor: shape=(5,), dtype=float64, numpy=array([1., 1., 1., 1., 0.])>)


In [3]:
json_ = load(file_path)
json_['theta']

[[0.172792096032393, 0.4108090717505792, 0.16521853809169357],
 [-0.6515786158021805, 0.45267793333655887, 0.22318728618200565],
 [-0.2684766176801426, 0.29055905209817656, 0.18228619809303787],
 [0.147066248327763, 0.014211120657898394, 0.27335649330622347],
 [-0.36822704350083346, -0.08145497399652639, -0.24105965633998913],
 [0.2994231063173138, 0.019861053740829495, -0.1462283754825443],
 [-0.39095423117842104, -0.12859612030943535, 0.004071090259171754],
 [-0.1378014526496852, 0.6470319071991036, 0.5033621576528972],
 [-1.3555812394829843, -0.9445066229838364, -0.08738604602758097],
 [-0.21109520578817678, 0.10682149874930555, 0.10866096551128179],
 [1.0589193775255241, -0.5560103813461407, -0.18880250356349904],
 [1.0213858037461652, 0.32335149810092345, 0.33153168618813084],
 [-0.25700318584373144, -0.8240375854278263, 0.08373237211137056],
 [0.05450704391077377, -0.6136760271222871, -0.3416133308902811],
 [-0.036021839863613715, -0.4723758115303887, -0.049134983926108634],
 [0.

In [4]:
counts_1, counts_total = sufficient_statistics(json_['data'],json_['vocabulary'])

display(counts_1)

display(counts_total)

array([[  0, 626, 433, 461, 487, 539, 611, 564, 543, 511],
       [328,   0, 525, 468, 458, 538, 598, 662, 467, 483],
       [449, 565,   0, 517, 488, 538, 597, 618, 483, 499],
       [569, 609, 550,   0, 542, 548, 571, 570, 592, 532],
       [465, 428, 601, 571,   0, 606, 510, 609, 482, 611],
       [602, 640, 509, 593, 538,   0, 537, 504, 570, 545],
       [484, 439, 615, 559, 604, 553,   0, 581, 515, 535],
       [407, 637, 446, 399, 456, 541, 630,   0, 502, 439],
       [330, 186, 867, 661, 651, 527, 333, 610,   0, 765],
       [446, 551, 514, 512, 534, 535, 537, 556, 529,   0]])

array([[   0, 1057, 1108, 1096, 1129, 1131, 1091, 1135, 1119, 1142],
       [1115,    0, 1121, 1149, 1058, 1137, 1159, 1079, 1157, 1114],
       [1160, 1174,    0, 1144, 1052, 1132, 1121, 1119, 1069, 1114],
       [1074, 1092, 1120,    0, 1097, 1114, 1070, 1142, 1078, 1082],
       [1127, 1136, 1156, 1132,    0, 1122, 1096, 1131, 1105, 1187],
       [1035, 1158, 1102, 1159, 1125,    0, 1047, 1089, 1122, 1127],
       [1129, 1125, 1090, 1111, 1187, 1083,    0, 1128, 1108, 1041],
       [1052, 1155, 1137, 1103, 1098, 1139, 1102,    0, 1089, 1132],
       [1045, 1178, 1135, 1072, 1016, 1124, 1117, 1110,    0, 1139],
       [1041, 1165, 1073, 1082, 1110, 1133, 1079, 1041, 1124,    0]])

In [10]:
counts_1, counts_total = sufficient_statistics(json_['data'],json_['vocabulary'])

v = 'word_1'
w = 'word_4'



V = 10
vi, wi = json_['vocabulary'][v], json_['vocabulary'][w]
import tensorflow as tf

def sigmoid(x:float): # might be a good idea to have this in a common library.
  return 1 / (1 + np.exp(-x))
#rng = np.random.default_rng(seed=1)

theta = np.array(json_['theta'])
rho_v = theta[vi, :]       # $\rho_v = \theta_v$
alpha_w = theta[wi+V, :]   # $\alpha_w = \theta_{V + w}$
eta =  rho_v.dot(alpha_w)             # $\eta = \rho_v^T \alpha_w$
p = sigmoid(eta)                      # Run it through the link function $\sigma; p = \sigma(\eta)$

print(p)

print(counts_1[vi, wi] / counts_total[vi, wi])

0.44983698855343096
0.43289224952741023


In [29]:
vocabulary = json_['vocabulary']
counts_1, counts_total = sufficient_statistics(json_['data'],json_['vocabulary'])

words = list(vocabulary.keys())
theta = np.array(json_['theta'])

for v in words:
    for w in words:
        compare_statistics(v, w, theta, counts_1, counts_total, vocabulary)
        print()


word_0, word_1
from theta : 0.5900321544129115
observed   : 0.5922421948912016

word_0, word_2
from theta : 0.4087594550187507
observed   : 0.3907942238267148

word_0, word_3
from theta : 0.4257711175020554
observed   : 0.4206204379562044

word_0, word_4
from theta : 0.44808809978836833
observed   : 0.43135518157661645

word_0, word_5
from theta : 0.49343401419955407
observed   : 0.4765694076038904

word_0, word_6
from theta : 0.5648457859789539
observed   : 0.5600366636113657

word_0, word_7
from theta : 0.5095411532502823
observed   : 0.49691629955947136

word_0, word_8
from theta : 0.48284410561989294
observed   : 0.48525469168900803

word_0, word_9
from theta : 0.42943343465876277
observed   : 0.44746059544658495

word_1, word_0
from theta : 0.2721381997073125
observed   : 0.29417040358744395


word_1, word_2
from theta : 0.4534160308912674
observed   : 0.4683318465655665

word_1, word_3
from theta : 0.40382395518324604
observed   : 0.4073107049608355

word_1, word_4
from theta : 

In [26]:
w

'word_0'

In [40]:
json_['theta']

[[0.172792096032393, 0.4108090717505792, 0.16521853809169357],
 [-0.6515786158021805, 0.45267793333655887, 0.22318728618200565],
 [-0.2684766176801426, 0.29055905209817656, 0.18228619809303787],
 [0.147066248327763, 0.014211120657898394, 0.27335649330622347],
 [-0.36822704350083346, -0.08145497399652639, -0.24105965633998913],
 [0.2994231063173138, 0.019861053740829495, -0.1462283754825443],
 [-0.39095423117842104, -0.12859612030943535, 0.004071090259171754],
 [-0.1378014526496852, 0.6470319071991036, 0.5033621576528972],
 [-1.3555812394829843, -0.9445066229838364, -0.08738604602758097],
 [-0.21109520578817678, 0.10682149874930555, 0.10866096551128179],
 [1.0589193775255241, -0.5560103813461407, -0.18880250356349904],
 [1.0213858037461652, 0.32335149810092345, 0.33153168618813084],
 [-0.25700318584373144, -0.8240375854278263, 0.08373237211137056],
 [0.05450704391077377, -0.6136760271222871, -0.3416133308902811],
 [-0.036021839863613715, -0.4723758115303887, -0.049134983926108634],
 [0.

In [41]:
json_['theta'][1]

[-0.6515786158021805, 0.45267793333655887, 0.22318728618200565]

In [11]:

# 0.6931471805599453
#python simulate.py --save_path "test100k.json" --vocab_size 10 --dimensionality 3 --n_datapoints 100000 --seed 1 --sd 0.5
e = Embedding(vocabulary=set(json_['vocabulary'].keys()), dimensionality=3, lambda0=0.0)
estimate = map_estimate(e, data_generator=load_simulated_data_generator(file_path, batch_size=100), batch_size=100, N = 100000, evaluate=False, training_loss=True, epochs = 20)

11:45:39 [TRAIN] (map): Epoch 0
11:45:39 [TRAIN] (map): Epoch 0


c:\Users\isacb\AppData\Local\Programs\Python\Python39\lib\site-packages\probabilistic_word_embeddings\estimation.py:56: UserWarning: The arguments 'ws' and 'ns' are ignored when using 'data_generator'
  warnings.warn("The arguments 'ws' and 'ns' are ignored when using 'data_generator'")
  0% (0 of 1000) |                       | Elapsed Time: 0:00:00 ETA:  --:--:--
  0% (1 of 1000) |                       | Elapsed Time: 0:00:00 ETA:   0:02:33
  1% (13 of 1000) |                      | Elapsed Time: 0:00:00 ETA:   0:02:32
  2% (26 of 1000) |                      | Elapsed Time: 0:00:00 ETA:   0:00:11
  3% (38 of 1000) |                      | Elapsed Time: 0:00:00 ETA:   0:00:11
  5% (51 of 1000) |#                     | Elapsed Time: 0:00:00 ETA:   0:00:08
  6% (64 of 1000) |#                     | Elapsed Time: 0:00:00 ETA:   0:00:08
  7% (76 of 1000) |#                     | Elapsed Time: 0:00:00 ETA:   0:00:07
  8% (89 of 1000) |#                     | Elapsed Time: 0:00:00 ETA:   

11:45:40 [TRAIN] (map): Epoch 0 mean training loss after 250 batches: 0.6967267110060205
11:45:40 [TRAIN] (map): Epoch 0 mean training loss after 250 batches: 0.6967267110060205


 25% (254 of 1000) |#####                | Elapsed Time: 0:00:01 ETA:   0:00:04
 26% (266 of 1000) |#####                | Elapsed Time: 0:00:01 ETA:   0:00:04
 27% (279 of 1000) |#####                | Elapsed Time: 0:00:01 ETA:   0:00:04
 29% (292 of 1000) |######               | Elapsed Time: 0:00:01 ETA:   0:00:04
 30% (304 of 1000) |######               | Elapsed Time: 0:00:01 ETA:   0:00:04
 31% (317 of 1000) |######               | Elapsed Time: 0:00:02 ETA:   0:00:04
 33% (330 of 1000) |######               | Elapsed Time: 0:00:02 ETA:   0:00:04
 34% (342 of 1000) |#######              | Elapsed Time: 0:00:02 ETA:   0:00:04
 35% (355 of 1000) |#######              | Elapsed Time: 0:00:02 ETA:   0:00:03
 36% (368 of 1000) |#######              | Elapsed Time: 0:00:02 ETA:   0:00:03
 38% (380 of 1000) |#######              | Elapsed Time: 0:00:02 ETA:   0:00:03
 39% (393 of 1000) |########             | Elapsed Time: 0:00:02 ETA:   0:00:03
 40% (406 of 1000) |########            

11:45:42 [TRAIN] (map): Epoch 0 mean training loss after 500 batches: 0.6952879252487242
11:45:42 [TRAIN] (map): Epoch 0 mean training loss after 500 batches: 0.6952879252487242


...

 50% (507 of 1000) |##########           | Elapsed Time: 0:00:03 ETA:   0:00:02
 51% (519 of 1000) |##########           | Elapsed Time: 0:00:03 ETA:   0:00:02
 53% (532 of 1000) |###########          | Elapsed Time: 0:00:03 ETA:   0:00:02
 54% (545 of 1000) |###########          | Elapsed Time: 0:00:03 ETA:   0:00:02
 55% (557 of 1000) |###########          | Elapsed Time: 0:00:03 ETA:   0:00:02
 57% (570 of 1000) |###########          | Elapsed Time: 0:00:03 ETA:   0:00:02
 58% (583 of 1000) |############         | Elapsed Time: 0:00:03 ETA:   0:00:02
 59% (595 of 1000) |############         | Elapsed Time: 0:00:03 ETA:   0:00:02
 60% (608 of 1000) |############         | Elapsed Time: 0:00:03 ETA:   0:00:02
 62% (621 of 1000) |#############        | Elapsed Time: 0:00:03 ETA:   0:00:02
 63% (633 of 1000) |#############        | Elapsed Time: 0:00:03 ETA:   0:00:02
 64% (646 of 1000) |#############        | Elapsed Time: 0:00:03 ETA:   0:00:02
 65% (659 of 1000) |#############       

11:47:35 [TRAIN] (map): Epoch 19 mean training loss after 750 batches: 0.6926245921343516
11:47:35 [TRAIN] (map): Epoch 19 mean training loss after 750 batches: 0.6926245921343516


 76% (760 of 1000) |###############      | Elapsed Time: 0:00:04 ETA:   0:00:01
 77% (773 of 1000) |################     | Elapsed Time: 0:00:04 ETA:   0:00:01
 78% (785 of 1000) |################     | Elapsed Time: 0:00:04 ETA:   0:00:01
 79% (798 of 1000) |################     | Elapsed Time: 0:00:04 ETA:   0:00:01
 81% (811 of 1000) |#################    | Elapsed Time: 0:00:04 ETA:   0:00:01
 82% (823 of 1000) |#################    | Elapsed Time: 0:00:05 ETA:   0:00:01
 83% (836 of 1000) |#################    | Elapsed Time: 0:00:05 ETA:   0:00:01
 84% (849 of 1000) |#################    | Elapsed Time: 0:00:05 ETA:   0:00:00
 86% (861 of 1000) |##################   | Elapsed Time: 0:00:05 ETA:   0:00:00
 87% (874 of 1000) |##################   | Elapsed Time: 0:00:05 ETA:   0:00:00
 88% (887 of 1000) |##################   | Elapsed Time: 0:00:05 ETA:   0:00:00
 89% (899 of 1000) |##################   | Elapsed Time: 0:00:05 ETA:   0:00:00
 91% (912 of 1000) |################### 

11:47:37 [TRAIN] (map): Epoch 19 mean training loss after 1000 batches: 0.6926589857838628
11:47:37 [TRAIN] (map): Epoch 19 mean training loss after 1000 batches: 0.6926589857838628


100% (1000 of 1000) |####################| Elapsed Time: 0:00:06 Time:  0:00:06


14:33:44 [TRAIN] (map): Epoch 0
14:33:44 [TRAIN] (map): Epoch 0


c:\Users\isacb\AppData\Local\Programs\Python\Python39\lib\site-packages\probabilistic_word_embeddings\estimation.py:56: UserWarning: The arguments 'ws' and 'ns' are ignored when using 'data_generator'
  warnings.warn("The arguments 'ws' and 'ns' are ignored when using 'data_generator'")


ValueError: Embeddings for an object of type <class 'list'> cannot be fetched. Check that you provide the word(s) as a str or a list of strs, as well as that the words are all elements are in the embedding.

In [13]:
set(json_['vocabulary'].keys())

{'word_0',
 'word_1',
 'word_2',
 'word_3',
 'word_4',
 'word_5',
 'word_6',
 'word_7',
 'word_8',
 'word_9'}

In [21]:
e[next(load_simulated_data_generator(file_path))[0]]

ValueError: Embeddings for an object of type <class 'tuple'> cannot be fetched. Check that you provide the word(s) as a str or a list of strs, as well as that the words are all elements are in the embedding.

In [22]:
next(load_simulated_data_generator(file_path))

(<tf.Tensor: shape=(1,), dtype=string, numpy=array([b'word_4'], dtype=object)>,
 <tf.Tensor: shape=(1,), dtype=string, numpy=array([b'word_5'], dtype=object)>,
 <tf.Tensor: shape=(1,), dtype=int32, numpy=array([1])>)

In [24]:
import tensorflow

In [25]:
tensorflow.__version__

'2.11.0'